# Autocorrelation: lab notebook

**Statistical Modelling and Forecasting, Foundations of time series**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox
import statsmodels.api as sm
import matplotlib.pyplot as plt

### 1. Hand calculation for y = 2, 4, 6, 5, 3

In [ ]:
# 1. Hand calculation for y = 2, 4, 6, 5, 3
y = np.array([2, 4, 6, 5, 3], dtype=float)
d = y - y.mean()
den = np.sum(d ** 2)
r1 = np.sum(d[1:] * d[:-1]) / den
r2 = np.sum(d[2:] * d[:-2]) / den
print("r1 =", round(r1, 3), "| r2 =", round(r2, 3), "| statsmodels:", np.round(acf(y, nlags=2), 3))

### 2. Simulate three processes and read their correlograms

In [ ]:
# 2. Simulate three processes and read their correlograms
rng = np.random.default_rng(3)
n = 200
e = rng.normal(size=n + 1)
white = e[1:]
ar1 = np.zeros(n)
for t in range(1, n):
    ar1[t] = 0.8 * ar1[t - 1] + e[t + 1]
ma1 = e[1:] + 0.8 * e[:-1]
band = 1.96 / np.sqrt(n)
print("95% band:", round(band, 3))
for name, x in [("white noise", white), ("AR(1) 0.8", ar1), ("MA(1) 0.8", ma1)]:
    r = acf(x, nlags=6)
    print(f"{name:12s}", np.round(r[1:], 2), "| bars outside band (lags 1-6):", int(np.sum(np.abs(r[1:]) > band)))

### 3. Ljung-Box test on 10 lags (H0: white noise)

In [ ]:
# 3. Ljung-Box test on 10 lags (H0: white noise)
for name, x in [("white noise", white), ("AR(1) 0.8", ar1)]:
    lb = acorr_ljungbox(x, lags=[10])
    print(f"{name:12s} Q={lb['lb_stat'].iloc[0]:8.2f}  p={lb['lb_pvalue'].iloc[0]:.4f}")

### 4. A real series: monthly CO2 (weekly data resampled) has a very slow ACF

In [ ]:
# 4. A real series: monthly CO2 (weekly data resampled) has a very slow ACF
co2 = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
print("CO2 ACF at lags 1, 6, 12, 24:", np.round(acf(co2, nlags=24)[[1, 6, 12, 24]], 3))
sm.graphics.tsa.plot_acf(co2, lags=36); plt.show()

## Expected output

Running all cells above should print:

```
r1 = 0.1 | r2 = -0.6 | statsmodels: [ 1.   0.1 -0.6]
95% band: 0.139
white noise  [-0.02  0.03 -0.1   0.04 -0.01  0.03] | bars outside band (lags 1-6): 0
AR(1) 0.8    [0.77 0.6  0.45 0.37 0.3  0.27] | bars outside band (lags 1-6): 6
MA(1) 0.8    [ 0.48 -0.04 -0.08 -0.03  0.01  0.04] | bars outside band (lags 1-6): 1
white noise  Q=    4.28  p=0.9338
AR(1) 0.8    Q=  317.66  p=0.0000
CO2 ACF at lags 1, 6, 12, 24: [0.993 0.945 0.935 0.87 ]
```

## Practice exercises

Try these in the cells below.

1. Switch between white noise, AR(1) with φ = 0.8 and φ = −0.8. Describe the three correlograms.
2. Choose "Seasonal" with period 12. At which lags are the peaks? Change the period to 7 and check.
3. Generate white noise with n = 40 and count how many bars cross the band. Repeat with "New sample" several times.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
